# Проект. Обучение собтвенной LLM

В рамках проекта выполнена реализация двух ключевых этапов создания языковой модели: **предобучение (pretrain)** на корпусе русской литературы и **дообучение с учителем (SFT)** на инструктивном датасете.  
Цель — отработать на практике полный цикл обучения LLM: от сбора и предобработки данных до тонкой настройки модели и оценки качества генерации.

**План проекта:**

**Этап 1. Pretrain**

1. Скачать тексты из репозитория `RussianNovels/corpus` и объединить в единый датасет
2. Предобработать текст, удалить дубликаты, нормализовать пунктуацию
3. Разбить текст на чанки длиной 512 токенов с добавлением `<bos>`/`<eos>`
4. Обучить BPE-токенизатор на 3000 токенов
5. Токенизировать данные и поместить в `Dataset`
6. Инициализировать модель decoder-only
7. Обученить модель с помощью `Trainer` 
8. Добавить коллбэки для валидации на `test_prompts`
9. Сгенерировать ответы на тестовые промпты

**Этап 2. Post-train SFT**

1. Загрузить датасет `d0rj/alpaca-cleaned-ru`
2. Преобразовать данные в формат диалогов `user` - `assistant`
3. Загрузить базовую модель `Qwen2.5-0.5B`
4. Настроить `SFTTrainer`
5. Дообучить модель на инструктивном датасете
6. Сгенерировать ответы на контрольные вопросы

In [2]:
import os
import re
import random
from glob import glob
import torch
import numpy as np
from datasets import load_dataset, Dataset
from tokenizers import Tokenizer, models, trainers, pre_tokenizers, decoders, processors
from transformers import (
    TrainerCallback,
    PreTrainedTokenizerFast,
    LlamaConfig,
    LlamaForCausalLM,
    AutoTokenizer,
    AutoModelForCausalLM,
)

from trl import SFTTrainer, SFTConfig

import warnings
warnings.filterwarnings("ignore", message=".*IProgress not found.*")


In [3]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

set_seed(42)

## Этап 1. Pretrain

### Предобработка данных

На данном этапе поработаем с данными. Загружаем все текстовые файлы из папки `./RussianNovels/corpus`, удаляем дубликаты строк, очищаем от некириллических символов, нормализуем повторяющуюся пунктуацию. Разбиваем текст на чанки, добавляем в начало и конец каждого чанка специальные токены `<bos>` и `<eos>`. Чанки сохраняем в файл для дальнейшего использования.

In [25]:
def preprocess_texts(folder_path, output_file):
    # Загружаем все строки из всех файлов
    all_lines = []
    for file_path in glob(os.path.join(folder_path, "*.txt")):
        with open(file_path, 'r', encoding='utf-8') as f:
            all_lines.extend(f.readlines())

    # Очищаем каждую строку
    cleaned_pairs = []
    for line in all_lines:
        clean = re.sub(r'[^а-яА-ЯёЁ\s\.\,\!\?\;\:\(\)\"\'\-\–\—]', '', line)
        clean = re.sub(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f\xad]', '', clean)
        clean = re.sub(r'[ѣѳѵ]', '', clean, flags=re.IGNORECASE)
        clean = re.sub(r'\s+', ' ', clean).strip()
        if clean:
            cleaned_pairs.append((clean, line))
    # Удаляем дубликаты
    seen = set()
    unique_lines = []
    for clean, orig in cleaned_pairs:
        if clean not in seen:
            seen.add(clean)
            unique_lines.append(orig)

    # Склеиваем
    full_text = ' '.join(unique_lines)

    # Очистка от цифр и многоточий
    full_text = re.sub(r'\d+', '', full_text)
    full_text = re.sub(r'[\.…]{2,}', ' ', full_text)
    full_text = re.sub(r'!{2,}', '!', full_text)
    full_text = re.sub(r'\?{2,}', '?', full_text)
    full_text = re.sub(r',{2,}', ',', full_text)
    full_text = re.sub(r'\s+([.,!?;:])', r'\1', full_text)
    full_text = re.sub(r'\s+', ' ', full_text).strip()

    # Сохраняем очищенный текст в файл
    with open(output_file, 'w', encoding='utf-8') as f:
        f.write(full_text)

    # Возвращаем список с одним элементом
    return full_text

In [26]:
# Обработаем текст
texts = preprocess_texts("./RussianNovels/corpus", "./dataset/preprocessed_chunks.txt")

In [27]:
# Посмотрим на пример
texts[:150]

'КАПИТАНСКАЯ ДОЧКА Береги честь смолоду. Пословица. ГЛАВА I СЕРЖАНТ ГВАРДИИ -- Был бы гвардии он завтра ж капитан. -- Того не надобно; пусть в армии по'

### Токенизация

Далее необходимо создать собственный токенизатор на основе BPE с размером словаря 3000 токенов. Используем предтокенизацию `ByteLevel`, обучаем на предобработанных чанках.

In [28]:
# Создаём BPE токенизатор
tokenizer = Tokenizer(models.BPE())

# Предтокенизация
tokenizer.pre_tokenizer = pre_tokenizers.ByteLevel(add_prefix_space=False)

# Тренер с базовыми спецтокенами
trainer = trainers.BpeTrainer(vocab_size=3000, special_tokens=["<unk>", "<bos>", "<eos>", "<pad>"])

In [29]:
# Обучаем на созданном файле
tokenizer.train(["dataset/preprocessed_chunks.txt"], trainer=trainer)

In [30]:
# Добавляем постпроцессинг и декодер
tokenizer.post_processor = processors.ByteLevel(trim_offsets=False)
tokenizer.decoder = decoders.ByteLevel()

# Сохраняем
tokenizer.save("models/bpe_tokenizer.json")

In [31]:
# Проверим размер словаря
print(f"Размер словаря: {tokenizer.get_vocab_size()}")

Размер словаря: 3000


Теперь загрузим обученный токенизатор, преобразуем каждый чанк в последовательность токенов длиной 512. Создим `transformers.Dataset` с полями `input_ids`, `attention_mask`, `labels` для causal LM.

In [32]:
# Загружаем обученный BPE-токенизатор
tokenizer = PreTrainedTokenizerFast(
    tokenizer_file="models/bpe_tokenizer.json",
    bos_token="<bos>",
    eos_token="<eos>",
    pad_token="<pad>",
    unk_token="<unk>"
)

In [33]:
# Токенизируем весь текст
tokens = tokenizer.encode(texts, add_special_tokens=False)

In [34]:
# Параметры
chunk_size = 512
pad_id = tokenizer.pad_token_id

# Получаем ID специальных токенов
bos_id = tokenizer.convert_tokens_to_ids("<bos>")
eos_id = tokenizer.convert_tokens_to_ids("<eos>")

# Нарезаем на чанки с числами вместо строк
chunks = [
    [bos_id] + tokens[i:i+chunk_size] + [eos_id]
    for i in range(0, len(tokens), chunk_size)
]

# Создаём Dataset
dataset = Dataset.from_dict({"input_ids": chunks})
# Добавляем attention_mask
dataset = dataset.map(lambda x: {"attention_mask": [1] * len(x["input_ids"])})

dataset = dataset.map(lambda x: {"labels": [-100 if id == pad_id else id for id in x["input_ids"]]})
# Устанавливаем формат PyTorch
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

# Сохраняем в переменную для trainer
tokenized_dataset = dataset

# Проверка
print(f"Размер датасета: {len(tokenized_dataset)}")
print(f"Пример input_ids: {tokenized_dataset[0]['input_ids'][:20]}")

Map: 100%|██████████| 28554/28554 [00:08<00:00, 3465.27 examples/s]

Размер датасета: 28554
Пример input_ids: tensor([   1, 1721, 1489, 1465, 2362, 1860, 1489, 1200, 1734, 1721, 1489, 2402,
         343, 1745, 2569, 1721, 1489,  437,  201,  403])


### Обучение SFT

Далее инициализируем модель и выполним обучение с нуля. Цель - получить модель, которая будет понимать структуру языка.

In [35]:
# Инициализируем модель Llama
config = LlamaConfig(
    hidden_size=1024,
    intermediate_size=1536,
    num_hidden_layers=16,
    num_attention_heads=16,
    num_key_value_heads=8,
    vocab_size=tokenizer.vocab_size,
    max_position_embeddings=512,
    resid_pdrop=0.1, 
    embd_pdrop=0.1, 
    attn_pdrop=0.1
)

model = LlamaForCausalLM(config).cuda()
print(f"Количество параметров: {model.num_parameters():,}")

Количество параметров: 132,006,912


In [36]:
# Тестовые промпты
test_prompts = [
    "Все мысли, которые имеют огромные последствия",
    "Сила войска зависит от его духа",
    "Мысль о том, что он принес страдания",
    "Человек сознает себя свободным",
    "Что бы ни случилось, я всегда буду",
    "Любовь мешает смерти",
    "Нет, жизнь не кончена",
    "Всякая мысль, даже самая простая",
    "Война не любезность, а самое гадкое дело",
    "Чтобы жить честно"
]

Посмотрим, что выдает модель, если она не обучена.

In [22]:
def generate_llama_untrained(prompt, max_new_tokens=100):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True)

print("=== Llama (случайные веса, до pretrain) ===\n")
for q in test_prompts:
    print(f"Вопрос: {q}")
    print(f"Ответ: {generate_llama_untrained(q)}")
    print("-" * 50)

=== Llama (случайные веса, до pretrain) ===

Вопрос: Все мысли, которые имеют огромные последствия
Ответ: Все мысли, которые имеют огромные последствия докб докббб Пб Пб П н н П н П н несмотря н несмотря н несмотря П несмотря П несмотря П несмотря П несмотря П несмотря П несмотря П несмотря П П мн П мн П мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн мн наши казалось мн наши казалось мн наши казалось наши казалось наши казалось наши казалось наши казалось наши
--------------------------------------------------
Вопрос: Сила войска зависит от его духа
Ответ: Сила войска зависит от его духа показалось зал моякого моякогокогокогокогокогокого�ийий�ий нуж������ий нуж������ ж ж ж ж наз ж наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз наз н

In [37]:
# Очизаем кеш в случае повторных попыток обучения
torch.cuda.empty_cache()

In [38]:
# Коллбэк для генерации на тестовых промптах после каждой эпохи
class GenerateCallback(TrainerCallback):
    def __init__(self, tokenizer, prompts, max_new_tokens=50):
        self.tokenizer = tokenizer
        self.prompts = prompts
        self.max_new_tokens = max_new_tokens

    def on_epoch_end(self, args, state, control, model=None, **kwargs):
        if model is None:
            return
        model.eval()
        print(f"\n===== Генерация после эпохи {state.epoch} =====")
        for prompt in self.prompts:
            inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
            inputs.pop("token_type_ids", None)
            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=self.max_new_tokens,
                    do_sample=False,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )
            print(f"Промпт: {prompt}\nГенерация: {tokenizer.decode(outputs[0], skip_special_tokens=True)}\n")
        model.train()

# Аргументы обучения
sft_config = SFTConfig(
    output_dir="./models",
    learning_rate=1e-4,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    max_length=512,
    num_train_epochs=3,
    report_to="none",
    logging_steps=1000,
    weight_decay=0.01,
    fp16=True,
    dataloader_num_workers=4,
    dataloader_pin_memory=True,
    remove_unused_columns=False,
    save_strategy="no",
    packing=False
)

# Trainer
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=tokenized_dataset,
    processing_class=tokenizer,
    callbacks=[GenerateCallback(tokenizer, test_prompts)],
)

# Запуск обучения
print("Training device:", trainer.model.device)
trainer.train()

Truncating train dataset:   0%|          | 0/28554 [00:00<?, ? examples/s]

Truncating train dataset: 100%|██████████| 28554/28554 [00:00<00:00, 132266.81 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 3}.


Training device: cuda:0


Step,Training Loss
1000,4.823700
2000,3.760500
3000,3.483700
4000,3.317800
5000,3.175600



===== Генерация после эпохи 1.0 =====
Промпт: Все мысли, которые имеют огромные последствия
Генерация: Все мысли, которые имеют огромные последствия, были в этом роде. В этом случае, в том, что они были в этом, и что они, как и в этом, и не может быть, и не может быть, и не может быть, и не может быть, и

Промпт: Сила войска зависит от его духа
Генерация: Сила войска зависит от его духа. Война, в сущности, в чем-то, в сущности, в этом роде, в чем-то вроде, в сущности, в этом роде, в том, чтобы не было в

Промпт: Мысль о том, что он принес страдания
Генерация: Мысль о том, что он принес страдания. Он был очень рад, но не мог понять, что он был влюблен в его душе. Он был очень хорошо, но не мог понять, что он был влюблен в его душе. Он был очень хорошо, и он был очень

Промпт: Человек сознает себя свободным
Генерация: Человек сознает себя свободным ееия...............................................

Промпт: Что бы ни случилось, я всегда буду
Генерация: Что бы ни случилось, я всегда буд

TrainOutput(global_step=5355, training_loss=3.6751487183415033, metrics={'train_runtime': 759.8326, 'train_samples_per_second': 112.738, 'train_steps_per_second': 7.048, 'total_flos': 3.392969451031757e+16, 'train_loss': 3.6751487183415033, 'entropy': 3.351427927151532, 'num_tokens': 43858911.0, 'mean_token_accuracy': 0.3865545328234283, 'epoch': 3.0})

In [39]:
# Сохраним модель
trainer.save_model("./models/pretrain_sft")
# Сохраняем токенизатор
tokenizer.save_pretrained("./models/pretrain_sft")

('./models/pretrain_sft/tokenizer_config.json',
 './models/pretrain_sft/special_tokens_map.json',
 './models/pretrain_sft/tokenizer.json')

In [40]:
# Сгенерируем ответы на тестовые вопросы
def generate(model, tokenizer, prompt, max_new_tokens=100):
    model.eval()
    input_ids = tokenizer.encode(prompt, return_tensors="pt").to(model.device)
    generated = input_ids
    with torch.no_grad():
        for _ in range(max_new_tokens):
            outputs = model(input_ids=generated)
            next_token_logits = outputs.logits[:, -1, :]
            next_token = torch.argmax(next_token_logits, dim=-1, keepdim=True)
            generated = torch.cat((generated, next_token), dim=1)
            if next_token.item() == tokenizer.eos_token_id:
                break
    return tokenizer.decode(generated[0], skip_special_tokens=True)

for p in test_prompts:
    print(f"Prompt: {p}\nResponse:{generate(model, tokenizer, p)}\n")

Prompt: Все мысли, которые имеют огромные последствия
Response:Все мысли, которые имеют огромные последствия, и, наконец, чтобы не было никаких причин, и потому, что они не могли бы быть и не могут, потому что они не могли бы быть и не могут. Но если бы они были, то они не могли бы быть и не могут. Но, если бы они были, то это было бы не так, как бы они ни были. Итак, они не могли бы быть и не могут. Но это не могло бы быть иначе, как

Prompt: Сила войска зависит от его духа
Response:Сила войска зависит от его духа. Войтами. В Вайнделе. В Вешен. В Вешене. В Вешене. В Вешен. В Вешен. В В Вайс В. В. В. В. В..............................................

Prompt: Мысль о том, что он принес страдания
Response:Мысль о том, что он принес страдания, и что он не может не думать, что он не может не быть, что он не может не быть, что он не может быть, что он не может быть, что он не может быть, что он не может быть, что он не может быть, что он не может быть, что он не может быть, что он не может

**Выводы:**

- Модель со случайными весами в некоторых ответах выдает просто набор букв или символов.
- Начальный loss ≈ 4.8 → финальный ≈ 3.2.  
- Модель повторяет ключевые слова или промпт целиком, предложения не похожи на осмысленные.  
- Модель, в целом, сохраняет структуру русского языка, но встречаются артефакты.

## Этап 2. Post‑train SFT

### Инициализация модели

На втором этапе необходимо обучить базовую модель `Qwen2.5-0.5B` и генерировать ответы на русскоязычные инструктивные запросы.  

In [17]:
# Инициализируем модель
model_name = "Qwen/Qwen2.5-0.5B"          
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float32,
    device_map="auto"
)

# Добавим специальный токен и шаблон чата
tokenizer.pad_token = tokenizer.eos_token
tokenizer.chat_template = "{% for message in messages %}{% if message['role'] == 'user' %}{{ '<|im_start|>user\n' + message['content'] + '<|im_end|>\n' }}{% elif message['role'] == 'assistant' %}{{ '<|im_start|>assistant\n' + message['content'] + '<|im_end|>\n' }}{% endif %}{% endfor %}"

In [18]:
# Загрузим данные
ds = load_dataset("d0rj/alpaca-cleaned-ru", split="train")

In [19]:
# Соеберм датасет в формате чата
def examples_to_messages(examples):
    data = {'messages': []}
    for example in examples:
        user_content = example['instruction']
        if example['input']:
            user_content += '\n' + example['input']
        data['messages'].append([
            {'role': 'user', 'content': user_content},
            {'role': 'assistant', 'content': example['output']}
        ])
    return Dataset.from_dict(data)

ds = examples_to_messages(ds)

In [20]:
# Применим шаблон чата к датасету
def formatting_func(example):
    messages = example.get('messages', [])
    if not messages:
        return ""
    text = tokenizer.apply_chat_template(messages, tokenize=False)
    return text

# Применим функцию, удалив столбец для экономии памяти
ds = ds.map(lambda x: {"text": formatting_func(x)})
ds = ds.remove_columns(["messages"])

Map: 100%|██████████| 51760/51760 [00:05<00:00, 8860.60 examples/s]


In [21]:
# Тестовые запросы
questions = [
    "сколько планет в нашей солнечной системе?",
    "расскажи стих",
    "когда собирать крыжовник?",
    "Как быстро выучить новый язык?"
]

Посмотрим теперь как отвечает базовая модель без обучения.

In [22]:
def generate_qwen(prompt, max_new_tokens=100):
    model.eval()
    messages = [{"role": "user", "content": prompt}]
    formatted = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    input_ids = tokenizer.encode(formatted, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response

print("\n=== Qwen2.5-0.5B (базовая, без SFT) ===")
for q in questions:
    print(f"Вопрос: {q}")
    print(f"Ответ: {generate_qwen(q)}")
    print("-" * 50)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.



=== Qwen2.5-0.5B (базовая, без SFT) ===
Вопрос: сколько планет в нашей солнечной системе?
Ответ: сколько планет в нашей солнечной системе?(UIAlertAction::new("сколько планет в нашей солнечной системе?(UIAlertAction::new("сколько планет в нашей солнечной системе?(UIAlertAction::new("сколько планет в нашей солнечной системе?(UIAlertAction::new("сколько планет в нашей солнечной системе?(UIAlertAction::new("сколько планет в нашей солнечной системе?(UIAlertAction::new("сколько планет в
--------------------------------------------------
Вопрос: расскажи стих
Ответ: 1. The sun sets in the west, and the moon rises in the east.
2. The wind blows, and the rain falls, and the sun sets in the west.
3. The moon rises in the east, and the sun sets in the west.
4. The wind blows, and the rain falls, and the moon rises in the east.
5. The sun sets in the west, and the moon rises in the east.
6. The wind blows, and the rain falls,
--------------------------------------------------
Вопрос: когда собира

Базовая модель не слишком хорошо справляется с русским языком.

In [ ]:
# Параметры обучения
sft_config = SFTConfig(
    output_dir="./models",
    per_device_train_batch_size=8,      
    gradient_accumulation_steps=1,
    learning_rate=2e-5,
    weight_decay=0.01,
    num_train_epochs=3,
    max_length=256,                     
    logging_steps=5000,
    save_strategy="no",
    report_to="none",
    dataset_text_field="text",
    dataloader_num_workers=4,         
    packing=False,                    
    fp16=True,                         
)

# Трейнер
trainer = SFTTrainer(
    model=model,
    args=sft_config,
    train_dataset=ds,
    processing_class=tokenizer,
)

# Запускаем обучение
trainer.train()

Adding EOS to train dataset:   0%|          | 0/51760 [00:00<?, ? examples/s]

Truncating train dataset: 100%|██████████| 51760/51760 [00:00<00:00, 362759.71 examples/s]
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss
5000,1.406500
10000,1.069200
15000,0.846100


TrainOutput(global_step=19410, training_loss=1.0086211576466706, metrics={'train_runtime': 2855.5753, 'train_samples_per_second': 54.378, 'train_steps_per_second': 6.797, 'total_flos': 8.518725220042752e+16, 'train_loss': 1.0086211576466706, 'entropy': 0.7404678611866201, 'num_tokens': 27220641.0, 'mean_token_accuracy': 0.8252498478440741, 'epoch': 3.0})

In [ ]:
# Сохранение модели
trainer.save_model("./models/qwen_sft")
# Сохраняем токенизатор
tokenizer.save_pretrained("./models/qwen_sft")

In [ ]:
# Функция генерации ответов в формате чата
def generate(model, tokenizer, prompt, max_new_tokens=100, do_sample=True, temperature=0.7):

    model.eval()
    # Применяем шаблон чата
    messages = [{"role": "user", "content": prompt}]
    formatted_prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True 
    )
    input_ids = tokenizer.encode(formatted_prompt, return_tensors="pt").to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            input_ids,
            max_new_tokens=max_new_tokens,
            do_sample=do_sample,
            temperature=temperature,
            top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id
        )
        
    # Обрезаем промпт
    response = tokenizer.decode(outputs[0][input_ids.shape[1]:], skip_special_tokens=True)
    return response

# Выведем результаты
for q in questions:
    print(f"Prompt: {q}\nResponce: {generate(model, tokenizer, q)}\n{'-'*50}")

Prompt: сколько планет в нашей солнечной системе?
Responce: assistant
В нашей Солнечной системе восемь планет. Это:
1. Меркурий
2. Венера
3. Земля
4. Марс
5. Юпитер
6. Сатурн
7. Уран
8. Нептун.

--------------------------------------------------
Prompt: расскажи стих
Responce: assistant
В тишине ночи,
В тихом спокойствии,
Я смотрю, как луна сияет вверх,
Сияет так ярко и чисто.

Он сжимает землю,
Когда он проходит через воздух,
Полный свет и красоты,
Как небесное тело движется.

Он приносит сладкий восторг,
В этот мир
--------------------------------------------------
Prompt: когда собирать крыжовник?
Responce: assistant
Крыжовник обычно следует собирать в конце лета, когда листья становятся мягкими и золотисто-желтыми, а цветы распускаются. Однако конкретное время сбора крыжовника может варьироваться в зависимости от региона и типа используемой садовой техники. Для садовых маховиков, для example, можно собирать крыжовник в конце августа или
---------------------------------------------

**Результаты post‑train SFT:**  
- Loss снизился с 1.41 до 0.85 модель хорошо усвоила формат инструкций.  
- Генерация на тестовых вопросах:
  - *«сколько планет в нашей солнечной системе?»* правильный ответ с перечислением.  
  - *«расскажи стих»*модель сочиняет стихотворение на русском.  
  - *«когда собирать крыжовник?»* ответ осмысленный, хотя есть неточности.  
  - *«Как быстро выучить новый язык?»* даёт практические советы.  
- Модель отвечает на русском языке, сохраняет структуру диалога.

## Выводы

1. Собрали и очистили русские литературные тексты, убрали дубликаты, посторонние символы и лишние пробелы, после чего разбили текст на фрагменты с маркерами начала и конца.
2. Обучили собственный BPE‑токенизатор на 3000 слов.
3. Подготовили датасет для обучения. Каждый фрагмент превратили в последовательность из 512 токенов, добавив при необходимости паддинг или обрезав лишнее.
4. Обучили с нуля модель Llama (132 млн параметров) на 43 тысячах фрагментов, и она научилась продолжать фразы на русском языке.
5. После дополнительной настройки на инструктивных данных модель уже не просто продолжает текст, а отвечает на вопросы, сочиняет стихи и даёт полезные советы как настоящий ассистент.

